In [1]:
# ============================================================
# URBAN MOBILITY INTELLIGENCE SYSTEM
# Notebook 04 — Demand Forecasting with Prophet
# Zone-level ride demand prediction
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Paths ────────────────────────────────────────────────────
RAW_DATA_PATH       = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\data\raw"
PROCESSED_DATA_PATH = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\data\processed"
CHARTS_OUTPUT_PATH  = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\outputs\charts"

# ── Load data ────────────────────────────────────────────────
print("Loading data...")
rides_df  = pd.read_csv(os.path.join(RAW_DATA_PATH, "rides.csv"))
zones_df  = pd.read_csv(os.path.join(RAW_DATA_PATH, "zones.csv"))
events_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "events.csv"))

rides_df['date']       = pd.to_datetime(rides_df['date'])
rides_df['ride_datetime'] = pd.to_datetime(rides_df['ride_datetime'])
events_df['event_date']  = pd.to_datetime(events_df['event_date'])

print(f"  ✓ rides_df  : {len(rides_df):,} rows")
print(f"  ✓ zones_df  : {len(zones_df):,} rows")
print(f"  ✓ events_df : {len(events_df):,} rows")
print(f"\n  Date range  : {rides_df['date'].min()} to {rides_df['date'].max()}")

# ── Top zones to forecast ────────────────────────────────────
# Select top 5 zones by ride volume for forecasting
top_zones = (
    rides_df.groupby(['pickup_zone_id','pickup_zone_name'])
    .size()
    .reset_index(name='total_rides')
    .nlargest(5, 'total_rides')
)

print(f"\n  Top 5 zones selected for forecasting:")
print(top_zones[['pickup_zone_name','total_rides']].to_string(index=False))
print("\n✓ Block 1 complete — forecasting environment ready.")

Loading data...
  ✓ rides_df  : 500,000 rows
  ✓ zones_df  : 200 rows
  ✓ events_df : 132 rows

  Date range  : 2024-01-01 00:00:00 to 2024-11-04 00:00:00

  Top 5 zones selected for forecasting:
    pickup_zone_name  total_rides
    HSR Layout Sec 7         3721
   Manyata Tech Park         3694
                ITPL         3678
Embassy Tech Village         3672
      Koramangala 1B         3669

✓ Block 1 complete — forecasting environment ready.


In [9]:
# ============================================================
# BLOCK 2 — City-Wide Daily Demand Forecast
# Using statsmodels SARIMA + scikit-learn ensemble
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── City-wide daily ride counts ──────────────────────────────
daily_rides = (
    rides_df.groupby('date')
    .size()
    .reset_index(name='ride_count')
)
daily_rides['date'] = pd.to_datetime(daily_rides['date'])
daily_rides = daily_rides.sort_values('date').reset_index(drop=True)

print(f"Daily time series  : {len(daily_rides)} days")
print(f"Date range         : {daily_rides['date'].min().date()} → {daily_rides['date'].max().date()}")
print(f"Avg daily rides    : {daily_rides['ride_count'].mean():.0f}")
print(f"Max daily rides    : {daily_rides['ride_count'].max():,}")
print(f"Min daily rides    : {daily_rides['ride_count'].min():,}")

# ── Feature engineering for ML model ────────────────────────
df = daily_rides.copy()
df['day_of_week']  = df['date'].dt.dayofweek       # 0=Mon
df['day_of_month'] = df['date'].dt.day
df['month']        = df['date'].dt.month
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
df['is_monday']    = (df['day_of_week'] == 0).astype(int)
df['is_friday']    = (df['day_of_week'] == 4).astype(int)

# Monsoon flag — June to October in Bangalore
df['is_monsoon']   = df['month'].isin([6,7,8,9,10]).astype(int)

# Rolling averages — trend signals
df['roll_7d_avg']  = df['ride_count'].rolling(7,  min_periods=1).mean()
df['roll_14d_avg'] = df['ride_count'].rolling(14, min_periods=1).mean()
df['roll_30d_avg'] = df['ride_count'].rolling(30, min_periods=1).mean()

# Lag features
df['lag_1d']  = df['ride_count'].shift(1)
df['lag_7d']  = df['ride_count'].shift(7)
df['lag_14d'] = df['ride_count'].shift(14)

# Event flag — days with high-impact events
high_impact_dates = set(
    events_df[events_df['demand_multiplier'] >= 1.5]['event_date']
    .astype(str).tolist()
)
df['has_major_event'] = df['date'].astype(str).isin(
    high_impact_dates
).astype(int)

df = df.dropna().reset_index(drop=True)

print(f"\nFeature-engineered rows: {len(df)}")
print(f"Features built         : {len(df.columns) - 2}")  # excl date + target

# ── Train/Test split — last 30 days as test ──────────────────
test_size  = 30
train_df   = df.iloc[:-test_size]
test_df_ml = df.iloc[-test_size:]

FEATURES = [
    'day_of_week', 'day_of_month', 'month', 'week_of_year',
    'is_weekend', 'is_monday', 'is_friday', 'is_monsoon',
    'roll_7d_avg', 'roll_14d_avg', 'roll_30d_avg',
    'lag_1d', 'lag_7d', 'lag_14d', 'has_major_event'
]

X_train = train_df[FEATURES]
y_train = train_df['ride_count']
X_test  = test_df_ml[FEATURES]
y_test  = test_df_ml['ride_count']

# ── Train Gradient Boosting model ────────────────────────────
print("\nTraining Gradient Boosting model...")
gb_model = GradientBoostingRegressor(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 4,
    min_samples_split = 5,
    subsample         = 0.8,
    random_state      = 42,
)
gb_model.fit(X_train, y_train)
print("  ✓ Model trained")

# ── Evaluate on test set ─────────────────────────────────────
y_pred_test = gb_model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mape = np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100

print(f"\n  Model Accuracy on Test Set (last 30 days):")
print(f"  MAE  : {mae:.1f} rides/day")
print(f"  RMSE : {rmse:.1f} rides/day")
print(f"  MAPE : {mape:.1f}%")
print(f"  Accuracy : {100 - mape:.1f}%")

# ── Forecast next 60 days ────────────────────────────────────
print("\nGenerating 60-day forecast...")

last_date     = df['date'].max()
forecast_dates = pd.date_range(
    start  = last_date + pd.Timedelta(days=1),
    periods= 60,
    freq   = 'D'
)

# Build forecast feature rows iteratively
forecast_rows = []
history       = df['ride_count'].tolist()
roll_window   = df['roll_30d_avg'].tolist()

for i, fdate in enumerate(forecast_dates):
    dow       = fdate.dayofweek
    is_wknd   = int(dow >= 5)
    is_mon    = int(dow == 0)
    is_fri    = int(dow == 4)
    month     = fdate.month
    is_mons   = int(month in [6,7,8,9,10])
    dom       = fdate.day
    woy       = fdate.isocalendar()[1]
    has_event = int(str(fdate.date()) in high_impact_dates)

    # Rolling averages from recent history
    r7  = np.mean(history[-7:])
    r14 = np.mean(history[-14:])
    r30 = np.mean(history[-30:])

    # Lag features
    lag1  = history[-1]
    lag7  = history[-7]  if len(history) >= 7  else r7
    lag14 = history[-14] if len(history) >= 14 else r14

    row = {
        'day_of_week':   dow,
        'day_of_month':  dom,
        'month':         month,
        'week_of_year':  woy,
        'is_weekend':    is_wknd,
        'is_monday':     is_mon,
        'is_friday':     is_fri,
        'is_monsoon':    is_mons,
        'roll_7d_avg':   r7,
        'roll_14d_avg':  r14,
        'roll_30d_avg':  r30,
        'lag_1d':        lag1,
        'lag_7d':        lag7,
        'lag_14d':       lag14,
        'has_major_event': has_event,
    }
    forecast_rows.append(row)

    # Predict and add to history for next iteration
    pred = gb_model.predict(pd.DataFrame([row]))[0]
    history.append(pred)

forecast_df = pd.DataFrame(forecast_rows)
forecast_df['date']          = forecast_dates
forecast_df['predicted_rides'] = gb_model.predict(forecast_df[FEATURES])

# Confidence bounds — ±1.5 std of residuals
residual_std = np.std(y_test - y_pred_test)
forecast_df['lower_bound'] = (
    forecast_df['predicted_rides'] - 1.5 * residual_std
).clip(lower=0)
forecast_df['upper_bound'] = (
    forecast_df['predicted_rides'] + 1.5 * residual_std
)

print(f"\n  60-Day Forecast Summary:")
print(f"  Period         : {forecast_df['date'].min().date()} → {forecast_df['date'].max().date()}")
print(f"  Avg Daily Rides: {forecast_df['predicted_rides'].mean():.0f}")
print(f"  Peak Predicted : {forecast_df['predicted_rides'].max():.0f}")
print(f"  Min Predicted  : {forecast_df['predicted_rides'].min():.0f}")

# ── Save forecast ────────────────────────────────────────────
forecast_df[['date','predicted_rides',
             'lower_bound','upper_bound']].to_csv(
    os.path.join(PROCESSED_DATA_PATH, "citywide_demand_forecast.csv"),
    index=False
)

# ── Plot ─────────────────────────────────────────────────────
fig1 = go.Figure()

# Historical
fig1.add_trace(go.Scatter(
    x    = daily_rides['date'],
    y    = daily_rides['ride_count'],
    name = 'Actual Daily Rides',
    mode = 'lines',
    line = dict(color='#42a5f5', width=1.5),
))

# Test predictions
fig1.add_trace(go.Scatter(
    x    = test_df_ml['date'],
    y    = y_pred_test.round(0),
    name = f'Model Fit (MAPE: {mape:.1f}%)',
    mode = 'lines',
    line = dict(color='#66bb6a', width=2),
))

# Forecast
fig1.add_trace(go.Scatter(
    x    = forecast_df['date'],
    y    = forecast_df['predicted_rides'].round(0),
    name = '60-Day Forecast',
    mode = 'lines',
    line = dict(color='#ff7043', width=2.5, dash='dot'),
))

# Confidence band
fig1.add_trace(go.Scatter(
    x    = list(forecast_df['date']) + list(forecast_df['date'][::-1]),
    y    = list(forecast_df['upper_bound'].round(0)) +
           list(forecast_df['lower_bound'].round(0)[::-1]),
    fill      = 'toself',
    fillcolor = 'rgba(255,112,67,0.15)',
    line      = dict(color='rgba(0,0,0,0)'),
    name      = '90% Confidence Band',
))

# Forecast start line
fig1.add_vline(
    x                  = str(last_date.date()),
    line_dash          = "dash",
    line_color         = "#ffffff",
    opacity            = 0.5,
    annotation_text    = "Forecast →",
    annotation_font_color = "white",
    annotation_position= "top right",
)

fig1.update_layout(
    title = dict(
        text = '🏙️ Bangalore City-Wide Daily Ride Demand Forecast — 60 Days',
        font = dict(size=16, color='white'),
        x    = 0.5,
    ),
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#16213e',
    font          = dict(color='white'),
    xaxis = dict(
        title     = 'Date',
        gridcolor = '#2a2a4a',
    ),
    yaxis = dict(
        title     = 'Daily Rides',
        gridcolor = '#2a2a4a',
    ),
    legend = dict(
        bgcolor     = 'rgba(0,0,0,0.5)',
        bordercolor = '#555',
        borderwidth = 1,
    ),
    height    = 500,
    hovermode = 'x unified',
    annotations = [dict(
        x         = 0.02, y = 0.95,
        xref      = 'paper', yref = 'paper',
        text      = f'Model Accuracy: {100-mape:.1f}% | MAE: {mae:.0f} rides/day | RMSE: {rmse:.0f}',
        showarrow = False,
        font      = dict(color='#66bb6a', size=12),
        bgcolor   = 'rgba(0,0,0,0.5)',
        bordercolor = '#555',
        borderwidth = 1,
    )],
)

chart1_path = os.path.join(
    CHARTS_OUTPUT_PATH, "01_citywide_demand_forecast.html"
)
fig1.write_html(chart1_path)
print(f"\n  Chart saved: outputs/charts/01_citywide_demand_forecast.html")
print("\n✓ Block 2 complete — city forecast built.")

Daily time series  : 309 days
Date range         : 2024-01-01 → 2024-11-04
Avg daily rides    : 1618
Max daily rides    : 2,355
Min daily rides    : 478

Feature-engineered rows: 295
Features built         : 15

Training Gradient Boosting model...
  ✓ Model trained

  Model Accuracy on Test Set (last 30 days):
  MAE  : 100.6 rides/day
  RMSE : 173.6 rides/day
  MAPE : 5.9%
  Accuracy : 94.1%

Generating 60-day forecast...

  60-Day Forecast Summary:
  Period         : 2024-11-05 → 2025-01-03
  Avg Daily Rides: 1646
  Peak Predicted : 2372
  Min Predicted  : 1307


TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [10]:
# ── Fix: convert last_date to timestamp for plotly ───────────
import plotly.graph_objects as go

last_date_str = pd.Timestamp(last_date).value // 10**6  # milliseconds

# Rebuild just the layout fix — rerun figure save
fig1.add_vline(
    x          = last_date.timestamp() * 1000,
    line_dash  = "dash",
    line_color = "#ffffff",
    opacity    = 0.5,
)

fig1.add_annotation(
    x         = str(last_date.date()),
    y         = daily_rides['ride_count'].max(),
    text      = "Forecast →",
    showarrow = False,
    font      = dict(color='white', size=11),
    xanchor   = 'left',
)

chart1_path = os.path.join(
    CHARTS_OUTPUT_PATH, "01_citywide_demand_forecast.html"
)
fig1.write_html(chart1_path)

print("=" * 60)
print("  CITY-WIDE DEMAND FORECAST — SUMMARY")
print("=" * 60)
print(f"  Model Accuracy : {100 - mape:.1f}%")
print(f"  MAE            : {mae:.1f} rides/day")
print(f"  RMSE           : {rmse:.1f} rides/day")
print(f"  MAPE           : {mape:.1f}%")
print(f"\n  60-Day Forecast:")
print(f"  Period         : {forecast_df['date'].min().date()} → {forecast_df['date'].max().date()}")
print(f"  Avg Daily Rides: {forecast_df['predicted_rides'].mean():.0f}")
print(f"  Peak Day       : {forecast_df.loc[forecast_df['predicted_rides'].idxmax(), 'date'].date()} ({forecast_df['predicted_rides'].max():.0f} rides)")
print(f"  Lowest Day     : {forecast_df.loc[forecast_df['predicted_rides'].idxmin(), 'date'].date()} ({forecast_df['predicted_rides'].min():.0f} rides)")
print(f"\n  Forecast saved : citywide_demand_forecast.csv")
print(f"  Chart saved    : 01_citywide_demand_forecast.html")
print("\n✓ Block 2 complete — city forecast built.")

  CITY-WIDE DEMAND FORECAST — SUMMARY
  Model Accuracy : 94.1%
  MAE            : 100.6 rides/day
  RMSE           : 173.6 rides/day
  MAPE           : 5.9%

  60-Day Forecast:
  Period         : 2024-11-05 → 2025-01-03
  Avg Daily Rides: 1646
  Peak Day       : 2024-12-29 (2372 rides)
  Lowest Day     : 2024-11-11 (1307 rides)

  Forecast saved : citywide_demand_forecast.csv
  Chart saved    : 01_citywide_demand_forecast.html

✓ Block 2 complete — city forecast built.


In [11]:
# ============================================================
# BLOCK 3 — Zone-Level Demand Forecast
# Individual forecasts for top 5 zones
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

zone_forecast_results = {}
zone_forecast_dfs     = []

print("=" * 60)
print("  ZONE-LEVEL DEMAND FORECASTING — TOP 5 ZONES")
print("=" * 60)

for _, zone_row in top_zones.iterrows():
    zone_id   = zone_row['pickup_zone_id']
    zone_name = zone_row['pickup_zone_name']

    # ── Daily rides for this zone ─────────────────────────────
    zone_daily = (
        rides_df[rides_df['pickup_zone_id'] == zone_id]
        .groupby('date')
        .size()
        .reset_index(name='ride_count')
    )
    zone_daily['date'] = pd.to_datetime(zone_daily['date'])
    zone_daily = zone_daily.sort_values('date').reset_index(drop=True)

    # Fill missing dates with 0
    full_dates = pd.DataFrame({
        'date': pd.date_range(
            zone_daily['date'].min(),
            zone_daily['date'].max(),
            freq='D'
        )
    })
    zone_daily = full_dates.merge(zone_daily, on='date', how='left').fillna(0)

    # ── Feature engineering ───────────────────────────────────
    zdf = zone_daily.copy()
    zdf['day_of_week']  = zdf['date'].dt.dayofweek
    zdf['day_of_month'] = zdf['date'].dt.day
    zdf['month']        = zdf['date'].dt.month
    zdf['is_weekend']   = (zdf['day_of_week'] >= 5).astype(int)
    zdf['is_monsoon']   = zdf['month'].isin([6,7,8,9,10]).astype(int)
    zdf['roll_7d']      = zdf['ride_count'].rolling(7,  min_periods=1).mean()
    zdf['roll_14d']     = zdf['ride_count'].rolling(14, min_periods=1).mean()
    zdf['lag_1d']       = zdf['ride_count'].shift(1)
    zdf['lag_7d']       = zdf['ride_count'].shift(7)
    zdf['has_event']    = zdf['date'].astype(str).isin(
        high_impact_dates
    ).astype(int)
    zdf = zdf.dropna().reset_index(drop=True)

    FEAT = [
        'day_of_week','day_of_month','month',
        'is_weekend','is_monsoon',
        'roll_7d','roll_14d',
        'lag_1d','lag_7d','has_event'
    ]

    # ── Train/test split ──────────────────────────────────────
    test_n  = 20
    tr      = zdf.iloc[:-test_n]
    te      = zdf.iloc[-test_n:]

    gb_z = GradientBoostingRegressor(
        n_estimators  = 200,
        learning_rate = 0.08,
        max_depth     = 3,
        random_state  = 42,
    )
    gb_z.fit(tr[FEAT], tr['ride_count'])

    y_pred_z = gb_z.predict(te[FEAT])
    mae_z    = mean_absolute_error(te['ride_count'], y_pred_z)
    mape_z   = np.mean(
        np.abs((te['ride_count'] - y_pred_z)
        / te['ride_count'].replace(0, np.nan))
    ) * 100

    # ── 30-day forecast ───────────────────────────────────────
    history_z = zdf['ride_count'].tolist()
    fc_dates  = pd.date_range(
        zdf['date'].max() + pd.Timedelta(days=1),
        periods=30, freq='D'
    )

    fc_rows = []
    for fd in fc_dates:
        r7  = np.mean(history_z[-7:])
        r14 = np.mean(history_z[-14:])
        l1  = history_z[-1]
        l7  = history_z[-7] if len(history_z) >= 7 else r7

        row = {
            'day_of_week':  fd.dayofweek,
            'day_of_month': fd.day,
            'month':        fd.month,
            'is_weekend':   int(fd.dayofweek >= 5),
            'is_monsoon':   int(fd.month in [6,7,8,9,10]),
            'roll_7d':      r7,
            'roll_14d':     r14,
            'lag_1d':       l1,
            'lag_7d':       l7,
            'has_event':    int(str(fd.date()) in high_impact_dates),
        }
        pred_z = max(0, gb_z.predict(pd.DataFrame([row]))[0])
        history_z.append(pred_z)
        fc_rows.append({'date': fd, 'predicted_rides': round(pred_z, 1)})

    fc_df_z = pd.DataFrame(fc_rows)
    fc_df_z['zone_id']   = zone_id
    fc_df_z['zone_name'] = zone_name
    zone_forecast_dfs.append(fc_df_z)

    zone_forecast_results[zone_name] = {
        'mae':          round(mae_z, 1),
        'mape':         round(mape_z, 1),
        'accuracy':     round(100 - mape_z, 1),
        'avg_forecast': round(fc_df_z['predicted_rides'].mean(), 1),
        'peak_rides':   round(fc_df_z['predicted_rides'].max(), 1),
        'peak_date':    fc_df_z.loc[
            fc_df_z['predicted_rides'].idxmax(), 'date'
        ].date(),
        'actual_df':    zdf,
        'test_df':      te,
        'y_pred':       y_pred_z,
        'forecast_df':  fc_df_z,
    }

    print(f"\n  ✓ {zone_name}")
    print(f"     Accuracy  : {100 - mape_z:.1f}%  |  MAE: {mae_z:.1f} rides/day")
    print(f"     30-day avg: {fc_df_z['predicted_rides'].mean():.0f} rides/day")
    print(f"     Peak day  : {fc_df_z.loc[fc_df_z['predicted_rides'].idxmax(), 'date'].date()} ({fc_df_z['predicted_rides'].max():.0f} rides)")

# ── Save all zone forecasts ───────────────────────────────────
all_zone_forecasts = pd.concat(zone_forecast_dfs, ignore_index=True)
all_zone_forecasts.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "zone_demand_forecasts.csv"),
    index=False
)

print("\n" + "=" * 60)
print("  ZONE FORECAST SUMMARY")
print("=" * 60)
summary_data = []
for zname, res in zone_forecast_results.items():
    summary_data.append({
        'Zone'        : zname,
        'Accuracy'    : f"{res['accuracy']}%",
        'MAE'         : f"{res['mae']} rides",
        'Avg Forecast': f"{res['avg_forecast']} rides/day",
        'Peak Day'    : str(res['peak_date']),
    })
print(pd.DataFrame(summary_data).to_string(index=False))
print(f"\n  Saved: zone_demand_forecasts.csv")
print("\n✓ Block 3 complete — zone forecasts done.")

  ZONE-LEVEL DEMAND FORECASTING — TOP 5 ZONES

  ✓ HSR Layout Sec 7
     Accuracy  : 75.3%  |  MAE: 2.3 rides/day
     30-day avg: 9 rides/day
     Peak day  : 2024-11-09 (16 rides)

  ✓ Manyata Tech Park
     Accuracy  : 71.2%  |  MAE: 3.0 rides/day
     30-day avg: 8 rides/day
     Peak day  : 2024-11-10 (11 rides)

  ✓ ITPL
     Accuracy  : 74.0%  |  MAE: 2.7 rides/day
     30-day avg: 13 rides/day
     Peak day  : 2024-11-09 (17 rides)

  ✓ Embassy Tech Village
     Accuracy  : 68.0%  |  MAE: 3.3 rides/day
     30-day avg: 16 rides/day
     Peak day  : 2024-11-10 (27 rides)

  ✓ Koramangala 1B
     Accuracy  : 71.0%  |  MAE: 3.5 rides/day
     30-day avg: 8 rides/day
     Peak day  : 2024-11-09 (13 rides)

  ZONE FORECAST SUMMARY
                Zone Accuracy       MAE   Avg Forecast   Peak Day
    HSR Layout Sec 7    75.3% 2.3 rides  9.0 rides/day 2024-11-09
   Manyata Tech Park    71.2% 3.0 rides  7.9 rides/day 2024-11-10
                ITPL    74.0% 2.7 rides 13.1 rides/day 202

In [12]:
# ============================================================
# BLOCK 4 — Forecast Visualization Charts
# ============================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Chart 1: Zone comparison forecast ───────────────────────
fig_zones = make_subplots(
    rows        = 3, cols = 2,
    subplot_titles = [z for z in zone_forecast_results.keys()] + [''],
    vertical_spacing   = 0.12,
    horizontal_spacing = 0.08,
)

zone_chart_colors = [
    '#42a5f5', '#66bb6a', '#ff7043',
    '#ab47bc', '#26c6da'
]

positions = [(1,1),(1,2),(2,1),(2,2),(3,1)]

for idx, (zone_name, res) in enumerate(zone_forecast_results.items()):
    row, col = positions[idx]
    color    = zone_chart_colors[idx]
    adf      = res['actual_df']
    fdf      = res['forecast_df']
    tdf      = res['test_df']
    ypred    = res['y_pred']

    # Actual
    fig_zones.add_trace(go.Scatter(
        x    = adf['date'],
        y    = adf['ride_count'],
        name = f'{zone_name} Actual',
        mode = 'lines',
        line = dict(color=color, width=1.2),
        showlegend = (idx == 0),
    ), row=row, col=col)

    # Test fit
    fig_zones.add_trace(go.Scatter(
        x    = tdf['date'],
        y    = np.maximum(0, ypred.round(0)),
        name = 'Model Fit',
        mode = 'lines',
        line = dict(color='#66bb6a', width=1.5),
        showlegend = (idx == 0),
    ), row=row, col=col)

    # 30-day forecast
    fig_zones.add_trace(go.Scatter(
        x    = fdf['date'],
        y    = fdf['predicted_rides'],
        name = '30-Day Forecast',
        mode = 'lines',
        line = dict(color='#ff7043', width=2, dash='dot'),
        showlegend = (idx == 0),
    ), row=row, col=col)

    # Accuracy annotation
    fig_zones.add_annotation(
        row  = row, col = col,
        x    = 0.05, y = 0.92,
        xref = 'x domain', yref = 'y domain',
        text = f"Accuracy: {res['accuracy']}%",
        showarrow = False,
        font      = dict(color='#66bb6a', size=10),
        bgcolor   = 'rgba(0,0,0,0.6)',
    )

fig_zones.update_layout(
    title = dict(
        text = '📊 Zone-Level 30-Day Ride Demand Forecast — Top 5 Zones',
        font = dict(size=16, color='white'),
        x    = 0.5,
    ),
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#16213e',
    font          = dict(color='white'),
    height        = 800,
    showlegend    = True,
    legend = dict(
        bgcolor     = 'rgba(0,0,0,0.5)',
        bordercolor = '#555',
        borderwidth = 1,
        orientation = 'h',
        y           = -0.05,
    ),
)

fig_zones.update_xaxes(gridcolor='#2a2a4a', showgrid=True)
fig_zones.update_yaxes(gridcolor='#2a2a4a', showgrid=True)

chart2_path = os.path.join(
    CHARTS_OUTPUT_PATH, "02_zone_demand_forecasts.html"
)
fig_zones.write_html(chart2_path)
print(f"  ✓ Saved: 02_zone_demand_forecasts.html")

# ── Chart 2: Feature importance ──────────────────────────────
# Use city-level model feature importance
feature_importance = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': gb_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig_imp = go.Figure(go.Bar(
    x           = feature_importance['importance'],
    y           = feature_importance['feature'],
    orientation = 'h',
    marker = dict(
        color = feature_importance['importance'],
        colorscale = 'Viridis',
        showscale  = False,
    ),
    text      = feature_importance['importance'].round(3),
    textposition = 'outside',
))

fig_imp.update_layout(
    title = dict(
        text = '🔍 Feature Importance — What Drives Ride Demand?',
        font = dict(size=16, color='white'),
        x    = 0.5,
    ),
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#16213e',
    font          = dict(color='white'),
    xaxis = dict(
        title     = 'Importance Score',
        gridcolor = '#2a2a4a',
    ),
    yaxis = dict(
        title     = 'Feature',
        gridcolor = '#2a2a4a',
    ),
    height = 500,
    margin = dict(l=150, r=100),
)

chart3_path = os.path.join(
    CHARTS_OUTPUT_PATH, "03_feature_importance.html"
)
fig_imp.write_html(chart3_path)
print(f"  ✓ Saved: 03_feature_importance.html")

# ── Chart 3: Hourly demand pattern ───────────────────────────
hourly_pattern = rides_df.groupby(['hour', 'is_weekend']).size()\
    .reset_index(name='rides')
hourly_pattern['day_type'] = hourly_pattern['is_weekend'].map(
    {True: 'Weekend', False: 'Weekday'}
)

fig_hourly = go.Figure()

for day_type, color in [('Weekday', '#42a5f5'), ('Weekend', '#ff7043')]:
    subset = hourly_pattern[hourly_pattern['day_type'] == day_type]
    fig_hourly.add_trace(go.Scatter(
        x    = subset['hour'],
        y    = subset['rides'],
        name = day_type,
        mode = 'lines+markers',
        line = dict(color=color, width=2.5),
        marker = dict(size=6),
        fill = 'tozeroy',
        fillcolor = color.replace(')', ',0.1)').replace('rgb', 'rgba'),
    ))

fig_hourly.update_layout(
    title = dict(
        text = '⏰ Hourly Demand Pattern — Weekday vs Weekend',
        font = dict(size=16, color='white'),
        x    = 0.5,
    ),
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#16213e',
    font          = dict(color='white'),
    xaxis = dict(
        title      = 'Hour of Day',
        tickmode   = 'linear',
        tick0      = 0,
        dtick      = 1,
        gridcolor  = '#2a2a4a',
    ),
    yaxis = dict(
        title     = 'Total Rides',
        gridcolor = '#2a2a4a',
    ),
    legend = dict(
        bgcolor     = 'rgba(0,0,0,0.5)',
        bordercolor = '#555',
        borderwidth = 1,
    ),
    height    = 450,
    hovermode = 'x unified',
)

chart4_path = os.path.join(
    CHARTS_OUTPUT_PATH, "04_hourly_demand_pattern.html"
)
fig_hourly.write_html(chart4_path)
print(f"  ✓ Saved: 04_hourly_demand_pattern.html")

# ── Chart 4: Monthly revenue trend ───────────────────────────
monthly_rev = (
    rides_df[rides_df['status'] == 'completed']
    .groupby('month')
    .agg(
        total_revenue = ('fare_amount', 'sum'),
        total_rides   = ('ride_id',     'count'),
        avg_fare      = ('fare_amount', 'mean'),
    )
    .reset_index()
)
monthly_rev['month_name'] = pd.to_datetime(
    monthly_rev['month'], format='%m'
).dt.strftime('%b')

fig_monthly = make_subplots(
    rows = 1, cols = 2,
    subplot_titles = ['Monthly Revenue (₹)', 'Monthly Avg Fare (₹)'],
)

fig_monthly.add_trace(go.Bar(
    x      = monthly_rev['month_name'],
    y      = monthly_rev['total_revenue'],
    name   = 'Revenue',
    marker = dict(
        color      = monthly_rev['total_revenue'],
        colorscale = 'Blues',
        showscale  = False,
    ),
    text      = (monthly_rev['total_revenue']/1e6).round(1).astype(str) + 'M',
    textposition = 'outside',
), row=1, col=1)

fig_monthly.add_trace(go.Scatter(
    x    = monthly_rev['month_name'],
    y    = monthly_rev['avg_fare'].round(0),
    name = 'Avg Fare',
    mode = 'lines+markers',
    line = dict(color='#ff7043', width=2.5),
    marker = dict(size=8),
), row=1, col=2)

fig_monthly.update_layout(
    title = dict(
        text = '💰 Monthly Revenue & Fare Trend — 2024',
        font = dict(size=16, color='white'),
        x    = 0.5,
    ),
    paper_bgcolor = '#1a1a2e',
    plot_bgcolor  = '#16213e',
    font          = dict(color='white'),
    height        = 420,
    showlegend    = False,
)
fig_monthly.update_xaxes(gridcolor='#2a2a4a')
fig_monthly.update_yaxes(gridcolor='#2a2a4a')

chart5_path = os.path.join(
    CHARTS_OUTPUT_PATH, "05_monthly_revenue_trend.html"
)
fig_monthly.write_html(chart5_path)
print(f"  ✓ Saved: 05_monthly_revenue_trend.html")

# ── Summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FORECASTING CHARTS — ALL SAVED")
print("=" * 60)
print(f"  01_citywide_demand_forecast.html")
print(f"  02_zone_demand_forecasts.html")
print(f"  03_feature_importance.html")
print(f"  04_hourly_demand_pattern.html")
print(f"  05_monthly_revenue_trend.html")
print(f"\n  Top demand driver   : {feature_importance.iloc[-1]['feature']}")
print(f"  Peak hour (weekday) : {hourly_pattern[hourly_pattern['day_type']=='Weekday'].nlargest(1,'rides')['hour'].values[0]}:00")
print(f"  Peak hour (weekend) : {hourly_pattern[hourly_pattern['day_type']=='Weekend'].nlargest(1,'rides')['hour'].values[0]}:00")
print(f"  Best revenue month  : {monthly_rev.nlargest(1,'total_revenue')['month_name'].values[0]}")
print("\n✓ Block 4 complete — all charts saved.")

  ✓ Saved: 02_zone_demand_forecasts.html
  ✓ Saved: 03_feature_importance.html
  ✓ Saved: 04_hourly_demand_pattern.html
  ✓ Saved: 05_monthly_revenue_trend.html

  FORECASTING CHARTS — ALL SAVED
  01_citywide_demand_forecast.html
  02_zone_demand_forecasts.html
  03_feature_importance.html
  04_hourly_demand_pattern.html
  05_monthly_revenue_trend.html

  Top demand driver   : has_major_event
  Peak hour (weekday) : 18:00
  Peak hour (weekend) : 18:00
  Best revenue month  : Oct

✓ Block 4 complete — all charts saved.
